# MC1 Graph — Exploratory Analysis with Altair
Goal: understand the structure of the graph (node types, edge types, genres, etc.) before any modelling.

## 1. Load data

In [1]:
import json
from pathlib import Path
import pandas as pd
import altair as alt

# Allow larger datasets (Altair has a 5000-row default limit)
alt.data_transformers.disable_max_rows()

path = Path("MC1_release/MC1_graph.json")
with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Top-level keys : {list(data.keys())}")
print(f"Nodes          : {len(data['nodes']):,}")
print(f"Links          : {len(data['links']):,}")

Top-level keys : ['directed', 'multigraph', 'graph', 'nodes', 'links']
Nodes          : 17,412
Links          : 37,857


## 2. Build node & edge DataFrames

In [2]:
nodes_df = pd.json_normalize(data["nodes"])
links_df = pd.json_normalize(data["links"])

# Coerce release_date to numeric (some entries may be strings or missing)
if 'release_date' in nodes_df.columns:
    nodes_df['release_date'] = pd.to_numeric(nodes_df['release_date'], errors='coerce')

print("Node columns :", list(nodes_df.columns))
print("Edge columns :", list(links_df.columns))
print()
print(f"Nodes shape  : {nodes_df.shape}")
print(f"Edges shape  : {links_df.shape}")

Node columns : ['Node Type', 'name', 'single', 'release_date', 'genre', 'notable', 'id', 'written_date', 'stage_name', 'notoriety_date']
Edge columns : ['Edge Type', 'source', 'target', 'key']

Nodes shape  : (17412, 10)
Edges shape  : (37857, 4)


In [3]:
nodes_df.head()

,Node Type,name,single,release_date,genre,notable,id,written_date,stage_name,notoriety_date
0,Song,Breaking These Chains,True,2017.0,Oceanus Folk,True,0,NaN,NaN,NaN
1,Person,Carlos Duffy,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN
2,Person,Min Qin,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN
3,Person,Xiuying Xie,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN
4,RecordLabel,Nautical Mile Records,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN


In [4]:
links_df.head()

,Edge Type,source,target,key
0,InterpolatesFrom,0,1841,0
1,RecordedBy,0,4,0
2,PerformerOf,1,0,0
3,ComposerOf,1,16180,0
4,PerformerOf,2,0,0


## 3. Quick summary — what types of things are in the graph?

In [5]:
# Counts of each node type
node_type_counts = (
    nodes_df['Node Type']
    .value_counts(dropna=False)
    .reset_index()
)
node_type_counts.columns = ['node_type', 'count']
node_type_counts

,node_type,count
0,Person,11361
1,Song,3615
2,RecordLabel,1217
3,Album,996
4,MusicalGroup,223


In [6]:
# Counts of each edge type (column is usually 'Edge Type')
edge_type_col = 'Edge Type' if 'Edge Type' in links_df.columns else 'edge_type'

edge_type_counts = (
    links_df[edge_type_col]
    .value_counts(dropna=False)
    .reset_index()
)
edge_type_counts.columns = ['edge_type', 'count']
edge_type_counts

,edge_type,count
0,PerformerOf,13587
1,RecordedBy,3798
2,ComposerOf,3290
3,ProducerOf,3209
4,DistributedBy,3013
5,LyricistOf,2985
6,InStyleOf,2289
7,InterpolatesFrom,1574
8,LyricalReferenceTo,1496
9,CoverOf,1429


## 4. Node types — bar chart
First Altair chart: distribution of node types. Hover for exact counts.

In [16]:
alt.Chart(node_type_counts).mark_bar().encode(
    x=alt.X('count:Q', title='Number of nodes'),
    y=alt.Y('node_type:N', sort='-x', title=None),
    color=alt.Color('node_type:N', legend=None),
    tooltip=['node_type', 'count']
).properties(
    width=500, height=300,
    title='Distribution of node types'
)

alt.Chart(...)

## 5. Edge types — bar chart
Same idea for edges. Edge types often tell you what kinds of relationships exist (e.g. *PerformerOf*, *MemberOf*, *ProducerOf*).

In [9]:
alt.Chart(edge_type_counts).mark_bar().encode(
    x=alt.X('count:Q', title='Number of edges'),
    y=alt.Y('edge_type:N', sort='-x', title=None),
    color=alt.Color('edge_type:N', legend=None),
    tooltip=['edge_type', 'count']
).properties(
    width=500, height=300,
    title='Distribution of edge types'
)

alt.Chart(...)

## 6. Top genres
Genres are usually attached to *Song* / *Album* nodes. Let's see what dominates the dataset.

In [ ]:
genre_counts = (nodes_df['genre'].dropna().value_counts().head(20).reset_index()
)
genre_counts.columns = ['genre', 'count']

alt.Chart(genre_counts).mark_bar().encode(
    x=alt.X('count:Q', title='Number of works'),
    y=alt.Y('genre:N', sort='-x', title=None),
    tooltip=['genre', 'count']
).properties(
    width=500, height=400,
    title='Top 20 genres'
)

alt.Chart(...)

## 7. Genre × Node Type heatmap
Which **node types** carry which genres? A heatmap shows it in one glance.

In [11]:
# Drop rows where genre is missing
gn = nodes_df.dropna(subset=['genre']).copy()

# Keep only top 15 genres to stay readable
top15 = gn['genre'].value_counts().head(15).index.tolist()
gn = gn[gn['genre'].isin(top15)]

heat = (
    gn.groupby(['Node Type', 'genre'])
      .size()
      .reset_index(name='count')
)

alt.Chart(heat).mark_rect().encode(
    x=alt.X('Node Type:N', title=None),
    y=alt.Y('genre:N', sort='-x'),
    color=alt.Color('count:Q', scale=alt.Scale(scheme='blues')),
    tooltip=['Node Type', 'genre', 'count']
).properties(
    width=400, height=400,
    title='Genre distribution across node types'
)

alt.Chart(...)

In [18]:
alt.Chart(heat).mark_rect().encode(
    x=alt.X('Node Type:N', title=None),
    y=alt.Y('genre:N', sort='-x'),
    color=alt.Color('count:Q', scale=alt.Scale(scheme='blues', type='log')),
    tooltip=['Node Type', 'genre', 'count']
).properties(
    width=400, height=400,
    title='Genre distribution across node types'
)

alt.Chart(...)

## 8. Release date distribution
When were these works released? A histogram (using `.bin()` — your first Altair transformation!) reveals the timeline.

In [12]:
# Filter to works that have a valid year
year_df = nodes_df.dropna(subset=['release_date']).copy()
year_df = year_df[(year_df['release_date'] >= 1900) &
                  (year_df['release_date'] <= 2030)]

alt.Chart(year_df).mark_bar().encode(
    alt.X('release_date:Q').bin(maxbins=40).title('Release year'),
    alt.Y('count():Q').title('Number of works'),
    tooltip=['count()']
).properties(
    width=700, height=300,
    title='Release year distribution'
)

alt.Chart(...)

## 9. Release dates — split by node type
Same histogram but coloured by node type. This shows whether songs and albums follow the same timeline.

In [17]:
alt.Chart(year_df).mark_bar().encode(
    alt.X('release_date:Q').bin(maxbins=40).title('Release year'),
    alt.Y('count():Q').title('Number'),
    color='Node Type:N',
    tooltip=['Node Type', 'count()']
).properties(
    width=700, height=300,
    title='Release year distribution by node type'
)

alt.Chart(...)

## 10. Missing data report
Which columns have missing values? Quick sanity check.

In [14]:
missing = nodes_df.isna().sum().reset_index()
missing.columns = ['column', 'missing']
missing['pct'] = (missing['missing'] / len(nodes_df) * 100).round(1)
missing = missing.sort_values('missing', ascending=False)

alt.Chart(missing[missing['missing'] > 0]).mark_bar().encode(
    x=alt.X('pct:Q', title='% missing'),
    y=alt.Y('column:N', sort='-x', title=None),
    tooltip=['column', 'missing', 'pct']
).properties(
    width=500, height=300,
    title='Missing values per node attribute'
)

alt.Chart(...)

## Next steps
Now that we know:
- the **node types** present (and their counts)
- the **edge types** present
- the **top genres**
- the **release-year span**
- where data is **missing**

...we can move on to building the actual graph object and doing centrality / community detection.

In [15]:
import networkx as nx

G = nx.MultiDiGraph()

# add nodes with attributes
for _, row in nodes_df.iterrows():
    node_id = row["id"]
    attrs = row.drop(labels=["id"]).to_dict()
    G.add_node(node_id, **attrs)

# add edges with edge attributes
edge_attr_cols = [c for c in links_df.columns if c not in ["source", "target"]]
for _, row in links_df.iterrows():
    edge_attrs = {c: row[c] for c in edge_attr_cols}
    G.add_edge(row["source"], row["target"], **edge_attrs)

print(f"Graph built: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Directed: {G.is_directed()}, Multigraph: {G.is_multigraph()}")

Graph built: 17,412 nodes, 37,857 edges
Directed: True, Multigraph: True


In [19]:
links_df.head(10)

,Edge Type,source,target,key
0,InterpolatesFrom,0,1841,0
1,RecordedBy,0,4,0
2,PerformerOf,1,0,0
3,ComposerOf,1,16180,0
4,PerformerOf,2,0,0
5,ProducerOf,2,16180,0
6,PerformerOf,3,0,0
7,InterpolatesFrom,5,5088,0
8,InStyleOf,5,14332,0
9,InterpolatesFrom,5,11677,0


In [20]:
nodes_df.columns

Index(['Node Type', 'name', 'single', 'release_date', 'genre', 'notable', 'id',
       'written_date', 'stage_name', 'notoriety_date'],
      dtype='str')

In [21]:
links_df.columns

Index(['Edge Type', 'source', 'target', 'key'], dtype='str')

In [22]:
# Build a small lookup table from node IDs to useful attributes
node_lookup = nodes_df.set_index('id')[['name', 'Node Type', 'genre']].rename(
    columns={'name': 'node_name', 'Node Type': 'node_type'}
)

# Join source info
edges = links_df.copy()
edges = edges.merge(node_lookup.add_prefix('source_'),
                    left_on='source', right_index=True, how='left')
edges = edges.merge(node_lookup.add_prefix('target_'),
                    left_on='target', right_index=True, how='left')

edges.head()

,Edge Type,source,target,key,source_node_name,source_node_type,source_genre,target_node_name,target_node_type,target_genre
0,InterpolatesFrom,0,1841,0,Breaking These Chains,Song,Oceanus Folk,Ripples and Whispers,Song,Indie Folk
1,RecordedBy,0,4,0,Breaking These Chains,Song,Oceanus Folk,Nautical Mile Records,RecordLabel,NaN
2,PerformerOf,1,0,0,Carlos Duffy,Person,NaN,Breaking These Chains,Song,Oceanus Folk
3,ComposerOf,1,16180,0,Carlos Duffy,Person,NaN,Siege of Barcelona's Twilight,Song,Oceanus Folk
4,PerformerOf,2,0,0,Min Qin,Person,NaN,Breaking These Chains,Song,Oceanus Folk


A heatmap showing which node types connect to which through edges. This is the schema of your graph

In [ ]:
schema = (
    edges.groupby(['source_node_type', 'target_node_type'])
         .size()
         .reset_index(name='count')
)

alt.Chart(schema).mark_rect().encode(
    x=alt.X('target_node_type:N', title='Target type'),
    y=alt.Y('source_node_type:N', title='Source type'),
    color=alt.Color('count:Q', scale=alt.Scale(scheme='blues', type='log')),
    tooltip=['source_node_type', 'target_node_type', 'count']
).properties(
    width=400, height=400,
    title='Graph schema: which node types connect to which'
)

alt.Chart(...)

For each kind of edge, what kind of source node initiates it

In [24]:
edge_breakdown = (
    edges.groupby(['Edge Type', 'source_node_type'])
         .size()
         .reset_index(name='count')
)

alt.Chart(edge_breakdown).mark_bar().encode(
    x=alt.X('count:Q', title='Number of edges'),
    y=alt.Y('Edge Type:N', sort='-x', title=None),
    color='source_node_type:N',
    tooltip=['Edge Type', 'source_node_type', 'count']
).properties(
    width=600, height=400,
    title='Edge types broken down by source node type'
)

alt.Chart(...)

Who has the most outgoing edges? Usually this means people who performed/produced/wrote a lot , (out degree)

In [27]:
top_sources = (
    edges.groupby(['source', 'source_node_name', 'source_node_type'])
         .size()
         .reset_index(name='out_degree')
         .sort_values('out_degree', ascending=False)
         .head(15)
)

alt.Chart(top_sources).mark_bar().encode(
    x=alt.X('out_degree:Q', title='Outgoing edges'),
    y=alt.Y('source_node_name:N', sort='-x', title=None),
    color='source_node_type:N',
    tooltip=['source_node_name', 'source_node_type', 'out_degree']
).properties(
    width=600, height=500,
    title='Top 15 most-connected source nodes (out-degree)'
)

alt.Chart(...)

### lots of nodes point to

In [28]:
top_targets = (
    edges.groupby(['target', 'target_node_name', 'target_node_type'])
         .size()
         .reset_index(name='in_degree')
         .sort_values('in_degree', ascending=False)
         .head(20)
)

alt.Chart(top_targets).mark_bar().encode(
    x=alt.X('in_degree:Q', title='Incoming edges'),
    y=alt.Y('target_node_name:N', sort='-x', title=None),
    color='target_node_type:N',
    tooltip=['target_node_name', 'target_node_type', 'in_degree']
).properties(
    width=600, height=500,
    title='Top 20 most-referenced target nodes (in-degree)'
)

alt.Chart(...)

The top of the list is dominated by Record Labels (blue) — Echo Chamber Records leads with ~230 incoming edges, Kamiogawa Rhythms close behind (~160)

Record labels accumulate edges because every artist signed to them and every work released through them points to them → high in-degree by definitio

Songs (orange) appear mid-list — Whispers of Finality and Ripples and Whispers show up here too (~150), confirming they're not just influential but also heavily referenced overall 

The mix of RecordLabel and Song in the top 20 hints that degree alone conflates two very different roles: structural hubs (labels) vs. influential works (songs)


In [29]:
edges['Edge Type'].unique()

<StringArray>
[  'InterpolatesFrom',         'RecordedBy',        'PerformerOf',
         'ComposerOf',         'ProducerOf',          'InStyleOf',
 'LyricalReferenceTo',            'CoverOf',      'DistributedBy',
           'MemberOf',         'LyricistOf',    'DirectlySamples']
Length: 12, dtype: str

In [32]:
influence = edges[edges['Edge Type'].isin(['InStyleOf', 'CoverOf'])]

# Most influential = most pointed-to by influence edges
most_influential = (
    influence.groupby(['target_node_name', 'target_node_type'])
             .size()
             .reset_index(name='times_cited')
             .sort_values('times_cited', ascending=False)
             .head(15)
)

alt.Chart(most_influential).mark_bar().encode(
    x='times_cited:Q',
    y=alt.Y('target_node_name:N', sort='-x'),
    color='target_node_type:N',
    tooltip=['target_node_name', 'target_node_type', 'times_cited']
).properties(width=600, height=400, title='Most influential nodes')

alt.Chart(...)

"Ripples and Whispers" and "Whispers of Finality" are the dominant influences in this graph (~65 citations each) — almost double the next contender

Almost everything is a Song (orange) — only one Album appears (Sweet Katharina's Lament, ~19). Albums almost never get cited as influences in this graph; that role belongs to individual tracks

The two top works share the word "Whispers" in their titles — possibly stylistic siblings or part of a movement worth digging into

Investigate the 'Whispers' cluster — are those 2 top influences related

In [ ]:
# pull the two nodes
whispers = nodes_df[nodes_df['name'].isin([
    'Ripples and Whispers',
    'Whispers of Finality'
])]
whispers

,Node Type,name,single,release_date,genre,notable,id,written_date,stage_name,notoriety_date
1841,Song,Ripples and Whispers,True,2010.0,Indie Folk,True,1841,NaN,NaN,NaN
15622,Song,Whispers of Finality,False,1997.0,Doom Metal,True,15622,NaN,NaN,NaN


In [ ]:
# Are they directly connected?
ids = whispers['id'].tolist()
direct = edges[
    (edges['source'].isin(ids) & edges['target'].isin(ids))
]
direct[['source_node_name', 'Edge Type', 'target_node_name']]

,source_node_name,Edge Type,target_node_name


This tells you: "who did what for each song". If the same person/label shows up for both → very strong link.

In [ ]:
# Who created each? (incoming edges)
# All edges pointing TO either song
incoming = edges[edges['target'].isin(ids)].copy()

# Show creator-style edges
creators = incoming[
    incoming['Edge Type'].isin([
        'PerformerOf', 'ComposerOf', 'LyricistOf',
        'ProducerOf', 'RecordedBy', 'DistributedBy'
    ])
]

(creators
 .groupby(['target_node_name', 'Edge Type', 'source_node_name', 'source_node_type'])
 .size()
 .reset_index(name='n'))

,target_node_name,Edge Type,source_node_name,source_node_type,n
0,Ripples and Whispers,ComposerOf,Juan Gao,Person,1
1,Ripples and Whispers,LyricistOf,Guiying Lu,Person,1
2,Ripples and Whispers,PerformerOf,Chao Tan,Person,1
3,Ripples and Whispers,PerformerOf,Guiying Lu,Person,1
4,Ripples and Whispers,PerformerOf,Min Tao,Person,1
5,Ripples and Whispers,ProducerOf,Min Tao,Person,1
6,Whispers of Finality,ComposerOf,Peter Huff,Person,1
7,Whispers of Finality,PerformerOf,Brett Brown,Person,1
8,Whispers of Finality,PerformerOf,Brittany Meyers,Person,1
9,Whispers of Finality,PerformerOf,Corey Moody,Person,1


The most powerful signal. Find every node that connects to both songs .

In [36]:
neighbors_a = set(edges[edges['target'] == ids[0]]['source']) | \
              set(edges[edges['source'] == ids[0]]['target'])
neighbors_b = set(edges[edges['target'] == ids[1]]['source']) | \
              set(edges[edges['source'] == ids[1]]['target'])

shared = neighbors_a & neighbors_b
print(f"Shared neighbours: {len(shared)}")

# Show who they are
shared_df = nodes_df[nodes_df['id'].isin(shared)][
    ['id', 'name', 'Node Type', 'genre']
]
shared_df

Shared neighbours: 3


,id,name,Node Type,genre
4733,4733,Golden Hour's Farewell,Song,Oceanus Folk
7140,7140,Waves of Longing,Song,Desert Rock
7278,7278,Requiem for a Fading Self,Song,Synthwave


In [37]:
# Things INFLUENCED BY each song (= edges where the song is the source of an influence edge)
# In MC1, "InStyleOf" / "InfluencedBy" usually point FROM the new work TO the influence,
# so we want edges where target = our song
fans = edges[
    edges['target'].isin(ids) &
    edges['Edge Type'].isin(['InfluencedBy', 'InStyleOf', 'CoverOf', 'LyricalReferenceTo'])
].copy()

fan_counts = (
    fans.groupby(['target_node_name', 'source_node_type'])
        .size()
        .reset_index(name='n_fans')
)

alt.Chart(fan_counts).mark_bar().encode(
    x='n_fans:Q',
    y=alt.Y('source_node_type:N'),
    color='target_node_name:N',
    tooltip=['target_node_name', 'source_node_type', 'n_fans']
).properties(
    width=500, height=200,
    title="Who's influenced by each 'Whispers' song?"
)

alt.Chart(...)

In [38]:
# Joining fan edges to the release year of the FAN node (the one citing)
fan_years = fans.merge(
    nodes_df[['id', 'release_date']].rename(columns={'id': 'source'}),
    on='source', how='left'
)

fan_years = fan_years.dropna(subset=['release_date'])

alt.Chart(fan_years).mark_bar().encode(
    alt.X('release_date:Q').bin(maxbins=20).title('Year of influenced work'),
    y='count():Q',
    color='target_node_name:N',
    tooltip=['target_node_name', 'count()']
).properties(
    width=600, height=300,
    title='When did each song spawn its influence wave?'
)

alt.Chart(...)

In [39]:
ids_whispers = whispers['id'].tolist()
ids_shared = [4733, 7140, 7278]

bridge = edges[
    ((edges['source'].isin(ids_whispers)) & (edges['target'].isin(ids_shared))) |
    ((edges['target'].isin(ids_whispers)) & (edges['source'].isin(ids_shared)))
][['source_node_name', 'Edge Type', 'target_node_name']]

bridge

,source_node_name,Edge Type,target_node_name
11927,Golden Hour's Farewell,InStyleOf,Whispers of Finality
11928,Golden Hour's Farewell,InStyleOf,Ripples and Whispers
17105,Waves of Longing,LyricalReferenceTo,Ripples and Whispers
17107,Waves of Longing,LyricalReferenceTo,Whispers of Finality
17430,Requiem for a Fading Self,DirectlySamples,Ripples and Whispers
17452,Requiem for a Fading Self,LyricalReferenceTo,Whispers of Finality
